# TFT / Direction Classifier — MMT BTC Order Flow

Trains on **1-minute BTC data from BigQuery** (`mmt_btc_1m_v3`): OHLCV **plus** MMT
order-flow (volume delta by trade size, liquidations, order book skew/depth, open
interest, funding).

**Primary model = `DirectionClassifier`** (section 6): predicts `p(up)` with a
*calibrated* probability. The TFT regression (section 7, optional) is legacy — V1
predicted % magnitude and was confirmed unprofitable.

**The experiment**: set `USE_MMT_FEATURES = True` / `False`, run twice, compare
out-of-sample accuracy *and* calibration. That delta is the whole question.

### Data caveats baked into this notebook
- History ceiling **362 days** — `LOOKBACK_DAYS` must stay <= 360.
- Columns zero-filled before ~2026-01-12 are **excluded**: `tps_*`, `buy_vol`,
  `sell_vol`, `agg_delta`, `buy_trades`, `sell_trades`, `last_price`.
  Substitutes with full history: `candle_*_vol` (volume), `vd_b*` (delta).
- Price/candle/funding live on the `binancef` rows; flow on the
  `binancef:bybitf` aggregate rows. The loader joins them on `ts`.
- Timestamps are **UTC** (BTC trades 24/7 — no session features).

## 0. Configuration

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # Windows OpenMP duplicate-DLL workaround

# ===== CONFIGURE THESE =====
SYMBOL     = "BTC"
BQ_PROJECT = "trading-brains"
BQ_TABLE   = "trading-brains.market_microstructure.mmt_btc_1m_v3"

LOOKBACK_DAYS     = 360     # MMT ceiling is 362 — do not raise
USE_MMT_FEATURES  = True    # <<< THE ABLATION SWITCH. False = OHLCV-only baseline
USE_VD_COHORTS    = True    # True = retail/mid/whale (3), False = all 10 buckets

HORIZON           = 60      # predict direction this many minutes ahead
MAX_ENCODER_LENGTH = 60     # bars of history fed to the model

# Classifier hyperparameters
CLS_MAX_EPOCHS   = 30
CLS_BATCH_SIZE   = 256
CLS_LR           = 5e-4
CLS_D_MODEL      = 64
CLS_NHEAD        = 4
CLS_NUM_LAYERS   = 2
CLS_DROPOUT      = 0.1
CLS_PATIENCE     = 7

# Legacy TFT regression (slow on CPU — leave False unless you need it)
RUN_TFT = False
MAX_EPOCHS = 20
BATCH_SIZE = 64
LEARNING_RATE = 0.001
HIDDEN_SIZE = 64
ATTENTION_HEAD_SIZE = 4
DROPOUT = 0.1
HIDDEN_CONTINUOUS_SIZE = 32
PATIENCE = 10
MAX_PREDICTION_LENGTH = 60

GCS_BUCKET = "tft-for-trading-brains"
GCS_MODEL_PATH = f"tft_checkpoint_{SYMBOL}_latest.ckpt"

import torch
ACCELERATOR = "gpu" if torch.cuda.is_available() else "cpu"
RUN_TAG = f"{SYMBOL}_{'mmt' if USE_MMT_FEATURES else 'ohlcv'}_h{HORIZON}"

print(f"Training on : {ACCELERATOR}")
print(f"Run tag     : {RUN_TAG}")
print(f"Lookback    : {LOOKBACK_DAYS}d | horizon {HORIZON}m | MMT features: {USE_MMT_FEATURES}")

## 1. Install / Import Dependencies

In [ ]:
%pip install google-cloud-storage google-cloud-bigquery db-dtypes scikit-learn

In [ ]:
import time, gc
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from google.cloud import storage, bigquery

print(f"PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")

## 2. Load Data from BigQuery

Two-pass join. Deliberately **excludes** every column that is zero-filled before
~2026-01-12, so the full 360-day window stays usable.

In [ ]:
def fetch_mmt_1min_data(lookback_days: int = 360) -> pd.DataFrame:
    """Load 1-min BTC bars + order flow from BigQuery (mmt_btc_1m_v3)."""
    client = bigquery.Client(project=BQ_PROJECT)

    query = f"""
    WITH bf AS (
      SELECT * EXCEPT(rn) FROM (
        SELECT ts, open, high, low, close,
               candle_buy_vol, candle_sell_vol, candle_total_vol, candle_delta,
               candle_total_trades, mark_price, funding_rate,
               ROW_NUMBER() OVER (PARTITION BY ts ORDER BY ts) AS rn
        FROM `{BQ_TABLE}` WHERE exchange = 'binancef'
      ) WHERE rn = 1
    ),
    agg AS (
      SELECT * EXCEPT(rn) FROM (
        SELECT ts, vd_b2, vd_b3, vd_b4, vd_b5, vd_b6, vd_b7, vd_b8, vd_b9,
               vd_b10, vd_b11, liq_buy_vol, liq_sell_vol, net_liq, liq_total,
               skew_0p5pct, skew_1pct, ask_depth_1pct, bid_depth_1pct, oi_close,
               ROW_NUMBER() OVER (PARTITION BY ts ORDER BY ts) AS rn
        FROM `{BQ_TABLE}` WHERE exchange = 'binancef:bybitf'
      ) WHERE rn = 1
    )
    SELECT
      bf.ts AS timestamp,
      bf.open, bf.high, bf.low, bf.close,
      bf.candle_total_vol AS volume,
      bf.candle_delta, bf.candle_total_trades,
      bf.funding_rate, bf.mark_price,
      agg.vd_b2, agg.vd_b3, agg.vd_b4, agg.vd_b5, agg.vd_b6,
      agg.vd_b7, agg.vd_b8, agg.vd_b9, agg.vd_b10, agg.vd_b11,
      agg.liq_buy_vol, agg.liq_sell_vol, agg.net_liq, agg.liq_total,
      agg.skew_0p5pct, agg.skew_1pct, agg.ask_depth_1pct, agg.bid_depth_1pct,
      agg.oi_close
    FROM bf JOIN agg USING (ts)
    WHERE bf.close > 0
      AND bf.ts >= TIMESTAMP_SUB(
            (SELECT MAX(ts) FROM `{BQ_TABLE}`), INTERVAL {lookback_days} DAY)
    ORDER BY bf.ts
    """

    print(f"Querying BigQuery ({lookback_days}d)...")
    df = client.query(query).to_dataframe()
    df["timestamp"] = pd.to_datetime(df["timestamp"]).dt.tz_localize(None)  # naive UTC
    df = df.sort_values("timestamp").reset_index(drop=True)

    # Fold the 10 VD buckets into 3 cohorts (fewer, less collinear features)
    if USE_VD_COHORTS:
        df["vd_retail"] = df[["vd_b2", "vd_b3"]].sum(axis=1)
        df["vd_mid"]    = df[[f"vd_b{i}" for i in range(4, 10)]].sum(axis=1)
        df["vd_whale"]  = df[["vd_b10", "vd_b11"]].sum(axis=1)
        df = df.drop(columns=[f"vd_b{i}" for i in range(2, 12)])

    if not USE_MMT_FEATURES:
        df = df[["timestamp", "open", "high", "low", "close", "volume"]]

    print(f"Loaded {len(df):,} bars: {df['timestamp'].min()} -> {df['timestamp'].max()}")
    print(f"Raw columns: {len(df.columns)}")
    return df


df_raw = fetch_mmt_1min_data(LOOKBACK_DAYS)
df_raw.tail()

### 2b. Sanity check — zero-rate by month

Zeros are *not* nulls. Any column that is ~100% zero before 2026-01 is a
collection gap and must not become a feature. This should come back clean,
because the loader already excludes the known-bad columns.

In [ ]:
chk = df_raw.copy()
chk["month"] = chk["timestamp"].dt.to_period("M")
num_cols = [c for c in chk.columns if c not in ("timestamp", "month")]
zr = chk.groupby("month")[num_cols].apply(lambda g: (g == 0).mean())

suspect = []
for c in num_cols:
    early = zr[c].iloc[:5].mean()
    late  = zr[c].iloc[-5:].mean()
    if early > 0.95 and late < 0.5:
        suspect.append((c, early, late))

if suspect:
    print("!! COLLECTION-GAP SUSPECTS (near-100% zero early, populated late):")
    for c, e, l in suspect:
        print(f"   {c}: {e:.0%} zero early -> {l:.0%} late   <-- DROP THIS")
else:
    print("OK: no column shows the early-zero / late-populated gap pattern.")

print("\nNaN counts (non-zero only):")
nn_ = df_raw.isna().sum()
print(nn_[nn_ > 0] if (nn_ > 0).any() else "  none")

## 3. Feature Engineering

Two blocks:

1. `calculate_price_features` — the original technical indicators (unchanged logic).
2. `calculate_flow_features` — **new**. Converts the raw MMT columns into
   *stationary ratios* rather than levels.

Why ratios matter here: BTC ran ~126k -> ~64k across this window, and the
validation split is the last 20%, i.e. a price regime the model never trained on.
Any absolute-level feature (raw OI, raw depth, raw volume) is out-of-distribution
at test time and will silently degrade results. Ratios and z-scores are not.

In [ ]:
def calculate_price_features(df: pd.DataFrame) -> pd.DataFrame:
    """Technical indicators from OHLCV. Requires: timestamp, open, high, low, close, volume."""
    df = df.copy()

    # returns
    for p in [1, 5, 15, 30, 60]:
        df[f"returns_{p}m"] = df["close"].pct_change(p)

    # candle shape
    df["high_low_ratio"]   = df["high"] / df["low"]
    df["close_open_ratio"] = df["close"] / df["open"]
    df["high_close_ratio"] = df["high"] / df["close"]
    df["low_close_ratio"]  = df["low"] / df["close"]
    rng = (df["high"] - df["low"] + 1e-10)
    df["upper_shadow"] = (df["high"] - np.maximum(df["open"], df["close"])) / rng
    df["lower_shadow"] = (np.minimum(df["open"], df["close"]) - df["low"]) / rng

    # moving averages (as distances / slopes — stationary)
    for p in [5, 10, 20, 30]:
        sma = df["close"].rolling(p).mean()
        df[f"sma_{p}_slope"]     = sma.pct_change()
        df[f"close_to_sma_{p}"]  = (df["close"] - sma) / sma
    sma60 = df["close"].rolling(60).mean()
    df["close_to_sma_60"] = (df["close"] - sma60) / sma60
    sma120 = df["close"].rolling(120).mean()
    df["sma_120_slope"]    = sma120.pct_change()
    df["close_to_sma_120"] = (df["close"] - sma120) / sma120

    # MACD — normalised by price so it is scale-free
    ema12 = df["close"].ewm(span=12, adjust=False).mean()
    ema26 = df["close"].ewm(span=26, adjust=False).mean()
    macd  = ema12 - ema26
    df["macd_norm"] = macd / df["close"]
    df["macd_histogram_norm"] = (macd - macd.ewm(span=9, adjust=False).mean()) / df["close"]

    # volatility
    for p in [5, 10, 20, 60]:
        df[f"volatility_{p}"] = df["returns_1m"].rolling(p).std()

    # ATR — normalised by price
    hl = df["high"] - df["low"]
    hc = (df["high"] - df["close"].shift()).abs()
    lc = (df["low"]  - df["close"].shift()).abs()
    tr = np.maximum(hl, np.maximum(hc, lc))
    df["atr_14_norm"] = tr.rolling(14).mean() / df["close"]
    df["atr_60_norm"] = tr.rolling(60).mean() / df["close"]

    # Bollinger — position only (std level is non-stationary)
    for p in [20, 60]:
        sma = df["close"].rolling(p).mean()
        std = df["close"].rolling(p).std()
        df[f"bb_position_{p}"] = (df["close"] - sma) / (2 * std + 1e-10)
        df[f"bb_width_{p}"]    = std / sma

    # RSI
    def rsi(s, p):
        d = s.diff()
        up = d.where(d > 0, 0).rolling(p).mean()
        dn = (-d.where(d < 0, 0)).rolling(p).mean()
        return 100 - 100 / (1 + up / (dn + 1e-10))
    for p in [14, 20, 60]:
        df[f"rsi_{p}"] = rsi(df["close"], p)

    # stochastic
    for p in [14, 60]:
        ll = df["low"].rolling(p).min()
        hh = df["high"].rolling(p).max()
        df[f"stoch_k_{p}"] = 100 * (df["close"] - ll) / (hh - ll + 1e-10)
    df["stoch_d_14"] = df["stoch_k_14"].rolling(3).mean()

    # rate of change
    df["roc_10"] = df["close"].pct_change(10) * 100
    df["roc_20"] = df["close"].pct_change(20) * 100

    # volume — ratios only, never levels
    df["volume_change"]    = df["volume"].pct_change(1)
    df["volume_change_5m"] = df["volume"].pct_change(5)
    for p in [5, 10, 20, 60]:
        vs = df["volume"].rolling(p).mean()
        df[f"volume_ratio_{p}"] = df["volume"] / (vs + 1e-10)

    # MFI (bounded 0-100, stationary)
    def mfi(d, p):
        tp = (d["high"] + d["low"] + d["close"]) / 3
        mf = tp * d["volume"]
        pos = mf.where(tp > tp.shift(), 0).rolling(p).sum()
        neg = mf.where(tp < tp.shift(), 0).rolling(p).sum()
        return 100 - 100 / (1 + pos / (neg + 1e-10))
    df["mfi_14"] = mfi(df, 14)
    df["mfi_60"] = mfi(df, 60)

    # rolling VWAP distance (rolling, not cumulative — cumulative is non-stationary)
    for p in [20, 60]:
        vw = (df["volume"] * df["close"]).rolling(p).sum() / (df["volume"].rolling(p).sum() + 1e-10)
        df[f"close_to_vwap_{p}"] = (df["close"] - vw) / vw

    # OBV as a normalised slope, not a raw cumulative sum
    sign = np.sign(df["close"].diff()).fillna(0)
    obv = (sign * df["volume"]).cumsum()
    df["obv_slope_20"] = obv.diff(20) / (df["volume"].rolling(20).sum() + 1e-10)

    # time-of-day (BTC is 24/7 — cyclic only, no session flags)
    hour = df["timestamp"].dt.hour
    minute = df["timestamp"].dt.minute
    dow = df["timestamp"].dt.dayofweek
    df["hour_sin"]   = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"]   = np.cos(2 * np.pi * hour / 24)
    df["minute_sin"] = np.sin(2 * np.pi * minute / 60)
    df["minute_cos"] = np.cos(2 * np.pi * minute / 60)
    df["day_sin"]    = np.sin(2 * np.pi * dow / 7)
    df["day_cos"]    = np.cos(2 * np.pi * dow / 7)

    return df


def calculate_flow_features(df: pd.DataFrame) -> pd.DataFrame:
    """MMT order-flow -> stationary ratios. No-op if MMT columns absent."""
    df = df.copy()
    if "oi_close" not in df.columns:
        return df                      # OHLCV-only baseline

    vol = df["volume"] + 1e-10

    # --- volume delta by cohort, normalised by volume -> [-1, 1] ---
    if USE_VD_COHORTS:
        for c in ["vd_retail", "vd_mid", "vd_whale"]:
            df[f"{c}_ratio"] = df[c] / vol
        # the interesting one: are whales and retail on opposite sides?
        df["vd_whale_vs_retail"] = df["vd_whale_ratio"] - df["vd_retail_ratio"]
        df = df.drop(columns=["vd_retail", "vd_mid", "vd_whale"])
    else:
        for i in range(2, 12):
            df[f"vd_b{i}_ratio"] = df[f"vd_b{i}"] / vol
        df = df.drop(columns=[f"vd_b{i}" for i in range(2, 12)])

    # --- aggregate taker delta ---
    df["candle_delta_ratio"] = df["candle_delta"] / vol

    # --- average trade size, vs its own baseline ---
    ats = df["volume"] / (df["candle_total_trades"] + 1e-10)
    df["avg_trade_size_ratio"] = ats / (ats.rolling(60).mean() + 1e-10)

    # --- liquidations: intensity vs baseline, and side imbalance ---
    df["liq_intensity"] = df["liq_total"] / (df["liq_total"].rolling(240).mean() + 1e-10)
    df["liq_imbalance"] = df["net_liq"] / (df["liq_total"] + 1e-10)     # +ve = forced selling
    df["liq_vs_volume"] = df["liq_total"] / vol

    # --- open interest: change and distance from its own mean ---
    df["oi_change_1m"]  = df["oi_close"].pct_change(1)
    df["oi_change_15m"] = df["oi_close"].pct_change(15)
    oi_sma = df["oi_close"].rolling(240).mean()
    df["oi_vs_sma"] = (df["oi_close"] - oi_sma) / (oi_sma + 1e-10)

    # --- order book: imbalance is already a ratio; depth normalised to baseline ---
    tot_depth = df["ask_depth_1pct"] + df["bid_depth_1pct"] + 1e-10
    df["depth_imbalance"] = (df["bid_depth_1pct"] - df["ask_depth_1pct"]) / tot_depth
    df["depth_vs_sma"]    = tot_depth / (tot_depth.rolling(240).mean() + 1e-10)

    # --- funding: level is already tiny; add a z-score for regime ---
    fr = df["funding_rate"]
    df["funding_z"] = (fr - fr.rolling(480).mean()) / (fr.rolling(480).std() + 1e-10)

    # --- mark/last dislocation in basis points ---
    df["mark_dislocation_bps"] = (df["close"] - df["mark_price"]) / df["close"] * 1e4

    # drop the raw level columns we just normalised away
    df = df.drop(columns=[c for c in [
        "candle_delta", "candle_total_trades", "liq_buy_vol", "liq_sell_vol",
        "net_liq", "liq_total", "oi_close", "ask_depth_1pct", "bid_depth_1pct",
        "mark_price",
    ] if c in df.columns])

    return df

In [ ]:
def remove_highly_correlated_features(df, threshold=0.95, protect=()):
    """Drop features with pairwise |corr| > threshold (keeps the first of each pair)."""
    exclude = set(["timestamp", "time_idx", "group", "target_up",
                   "open", "high", "low", "close", "volume"]) | set(protect)
    feats = [c for c in df.columns
             if c not in exclude and pd.api.types.is_numeric_dtype(df[c])]

    corr = df[feats].dropna().corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [c for c in upper.columns if any(upper[c] > threshold)]

    print(f"Correlation filter (>{threshold}):")
    print(f"  before: {len(feats)}  dropped: {len(to_drop)}  remaining: {len(feats)-len(to_drop)}")
    if to_drop:
        print(f"  dropped -> {to_drop}")
    return df.drop(columns=to_drop)


def prepare_dataset(df: pd.DataFrame) -> pd.DataFrame:
    df = calculate_price_features(df)
    df = calculate_flow_features(df)

    # label: did close rise over the next HORIZON minutes?
    df["target_up"] = (df["close"].shift(-HORIZON) > df["close"]).astype("float32")
    df.loc[df.index[-HORIZON:], "target_up"] = np.nan      # unknowable at the tail

    # clean
    df = df.replace([np.inf, -np.inf], np.nan)
    feat_cols = [c for c in df.columns if c not in ("timestamp", "target_up")]
    df = df.dropna(subset=feat_cols)          # drops indicator warm-up rows
    df = df.dropna(subset=["target_up"])      # drops the unknowable tail
    df = df.reset_index(drop=True)

    df = remove_highly_correlated_features(df, threshold=0.95)

    df["time_idx"] = range(len(df))
    df["group"] = "default"
    print(f"\nPrepared: {len(df):,} rows, {len(df.columns)} columns")
    return df


df_all = prepare_dataset(df_raw)
data_start, data_end = str(df_all["timestamp"].min()), str(df_all["timestamp"].max())
print(f"Range: {data_start} -> {data_end}")
print(f"Base rate (share of UP labels): {df_all['target_up'].mean():.1%}")
del df_raw; gc.collect()
df_all.head()

## 4. Chronological Split + Scaling

Split is **chronological** (never random — adjacent minutes leak heavily).
The scaler is fit on **train only** and applied to validation: fitting on
everything would leak validation statistics into training.

In [ ]:
EXCLUDE_FROM_FEATURES = [
    "timestamp", "time_idx", "group", "target_up",
    # absolute price/volume levels: non-stationary across the 126k->64k regime shift
    "open", "high", "low", "close", "volume",
]
feature_cols = [c for c in df_all.columns
                if c not in EXCLUDE_FROM_FEATURES and pd.api.types.is_numeric_dtype(df_all[c])]

flow_cols = [c for c in feature_cols if any(k in c for k in
             ("vd_", "liq_", "oi_", "depth", "funding", "mark_disl",
              "candle_delta", "avg_trade_size"))]
print(f"Total features : {len(feature_cols)}")
print(f"  price/technical: {len(feature_cols)-len(flow_cols)}")
print(f"  MMT order flow : {len(flow_cols)}  -> {flow_cols}")

cutoff = int(len(df_all) * 0.8)
df_train = df_all.iloc[:cutoff].reset_index(drop=True)
df_val   = df_all.iloc[cutoff:].reset_index(drop=True)

print(f"\nTrain: {len(df_train):,}  {df_train.timestamp.min().date()} -> {df_train.timestamp.max().date()}")
print(f"Val  : {len(df_val):,}  {df_val.timestamp.min().date()} -> {df_val.timestamp.max().date()}")
print(f"Base rate  train {df_train.target_up.mean():.1%} | val {df_val.target_up.mean():.1%}")

# --- scaling: fit on TRAIN ONLY ---
scaler = StandardScaler().fit(df_train[feature_cols].values)
df_train_s = df_train.copy(); df_val_s = df_val.copy()
df_train_s[feature_cols] = scaler.transform(df_train[feature_cols].values)
df_val_s[feature_cols]   = scaler.transform(df_val[feature_cols].values)
print("\nScaler fitted on train only and applied to both splits.")

## 5. Direction Classifier

Small transformer over a 60-bar window -> `p(up)` at `HORIZON` minutes.
Focal loss keeps it from collapsing onto the majority class.

In [ ]:
class DirectionDataset(Dataset):
    """(window of features, binary label). Window ENDS at the base bar -> no lookahead."""
    def __init__(self, df, feature_cols, encoder_length=60):
        self.features = df[feature_cols].values.astype("float32")
        self.labels   = df["target_up"].values.astype("float32")
        self.enc = encoder_length
        self.n = len(df) - encoder_length + 1

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        x = self.features[idx: idx + self.enc]      # last row = base bar
        y = self.labels[idx + self.enc - 1]         # label of that base bar
        return torch.from_numpy(x), torch.tensor(y)


class DirectionClassifier(pl.LightningModule):
    def __init__(self, n_features, d_model=64, nhead=4, num_layers=2,
                 dropout=0.1, learning_rate=5e-4, focal_alpha=0.5, focal_gamma=1.5):
        super().__init__()
        self.save_hyperparameters()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_embedding = nn.Parameter(torch.randn(1, 256, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation="gelu")
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model), nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_model // 2, 1))

    def forward(self, x):
        h = self.input_proj(x)
        h = h + self.pos_embedding[:, :h.size(1), :]
        h = self.transformer(h)
        return self.classifier(h[:, -1, :]).squeeze(-1)

    def focal_loss(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        a_t = self.hparams.focal_alpha * targets + (1 - self.hparams.focal_alpha) * (1 - targets)
        return (a_t * (1 - p_t) ** self.hparams.focal_gamma * bce).mean()

    def _step(self, batch, stage):
        x, y = batch
        logits = self(x)
        loss = self.focal_loss(logits, y)
        acc = ((torch.sigmoid(logits) > 0.5).float() == y).float().mean()
        self.log(f"{stage}_loss", loss, prog_bar=True)
        self.log(f"{stage}_acc", acc, prog_bar=True)
        return loss

    def training_step(self, b, i):   return self._step(b, "train")
    def validation_step(self, b, i): return self._step(b, "val")

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.hparams.learning_rate, weight_decay=0.01)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.trainer.max_epochs)
        return [opt], [sch]

print("DirectionClassifier defined.")

In [ ]:
train_ds = DirectionDataset(df_train_s, feature_cols, MAX_ENCODER_LENGTH)
val_ds   = DirectionDataset(df_val_s,   feature_cols, MAX_ENCODER_LENGTH)
print(f"Train windows: {len(train_ds):,} | Val windows: {len(val_ds):,}")

# NOTE: shuffle=True is fine here — each sample is a self-contained window and the
# train/val boundary is chronological. Shuffling only affects gradient order.
train_dl = DataLoader(train_ds, batch_size=CLS_BATCH_SIZE, shuffle=True,  num_workers=0, drop_last=True)
val_dl   = DataLoader(val_ds,   batch_size=CLS_BATCH_SIZE, shuffle=False, num_workers=0)

cls_model = DirectionClassifier(
    n_features=len(feature_cols), d_model=CLS_D_MODEL, nhead=CLS_NHEAD,
    num_layers=CLS_NUM_LAYERS, dropout=CLS_DROPOUT, learning_rate=CLS_LR)
print(f"Parameters: {sum(p.numel() for p in cls_model.parameters())/1e3:.1f}k")

cls_ckpt_dir = f"checkpoints/classifier_{RUN_TAG}"
os.makedirs(cls_ckpt_dir, exist_ok=True)
cls_ckpt = ModelCheckpoint(dirpath=cls_ckpt_dir, monitor="val_loss",
                           filename="cls-{epoch:02d}-{val_loss:.4f}", save_top_k=1, mode="min")

cls_trainer = pl.Trainer(
    max_epochs=CLS_MAX_EPOCHS,
    callbacks=[EarlyStopping(monitor="val_loss", patience=CLS_PATIENCE, mode="min", verbose=True),
               cls_ckpt, LearningRateMonitor(logging_interval="epoch")],
    accelerator=ACCELERATOR, gradient_clip_val=1.0, enable_model_summary=True)

print(f"Training {RUN_TAG} on {ACCELERATOR}...")
cls_trainer.fit(cls_model, train_dataloaders=train_dl, val_dataloaders=val_dl)

## 6. Evaluation — accuracy, calibration, regime stability

The three numbers that matter:

- **Accuracy vs base rate.** Beating 50% is not the bar; beating the *base rate*
  (the share of UP labels) is. A model that always says "up" scores the base rate.
- **Calibration (ECE / Brier).** When it says 70%, is it right ~70% of the time?
  This is what V1 never had and what makes confidence usable for sizing.
- **H1/H2 stability.** Validation split in half. An edge in only one half is regime luck.

In [ ]:
best_cls = DirectionClassifier.load_from_checkpoint(cls_ckpt.best_model_path)
best_cls.eval()

probs, labels = [], []
with torch.no_grad():
    for x, y in val_dl:
        probs.append(torch.sigmoid(best_cls(x)).cpu())
        labels.append(y.cpu())
probs  = torch.cat(probs).numpy()
labels = torch.cat(labels).numpy()

base_rate = labels.mean()
pred = (probs > 0.5).astype(int)
acc = (pred == labels).mean()
brier = float(np.mean((probs - labels) ** 2))

# Expected Calibration Error (10 equal-width bins)
ece, bins = 0.0, np.linspace(0, 1, 11)
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (probs >= lo) & (probs < hi)
    if m.sum():
        ece += m.mean() * abs(probs[m].mean() - labels[m].mean())

print("=" * 62)
print(f"RESULTS — {RUN_TAG}   (MMT features: {USE_MMT_FEATURES})")
print("=" * 62)
print(f"Validation windows : {len(labels):,}")
print(f"Base rate (always-up): {base_rate*100:.2f}%")
print(f"Accuracy           : {acc*100:.2f}%   ({(acc-base_rate)*100:+.2f} pp vs base rate)")
print(f"Brier score        : {brier:.4f}   (lower better; {np.mean((base_rate-labels)**2):.4f} = always-base-rate)")
print(f"ECE                : {ece:.4f}   (lower better; <0.05 is decent)")
print(f"Predicted UP share : {pred.mean()*100:.1f}%   (collapse warning if ~0% or ~100%)")

print("\nCalibration:")
print(f"  {'bucket':<12}{'n':>8}{'mean p(up)':>13}{'actual up%':>13}{'gap':>9}")
for lo, hi in [(0,.3),(.3,.4),(.4,.45),(.45,.5),(.5,.55),(.55,.6),(.6,.7),(.7,1.)]:
    m = (probs >= lo) & (probs < hi)
    if m.sum() < 20:
        continue
    mp, au = probs[m].mean(), labels[m].mean()
    print(f"  [{lo:.2f},{hi:.2f}){m.sum():>8,}{mp:>13.3f}{au*100:>12.1f}%{(au-mp)*100:>8.1f}")

# --- H1 / H2 regime stability ---
half = len(labels) // 2
print("\nRegime stability (validation split in half):")
for nm, sl in [("H1", slice(0, half)), ("H2", slice(half, None))]:
    p_, l_ = probs[sl], labels[sl]
    print(f"  {nm}: n={len(l_):,}  base={l_.mean()*100:.1f}%  "
          f"acc={((p_>0.5)==l_).mean()*100:.1f}%  "
          f"edge={(((p_>0.5)==l_).mean()-max(l_.mean(),1-l_.mean()))*100:+.1f}pp")

# --- does confidence rank accuracy? (V1's core failure) ---
print("\nAccuracy by confidence (|p-0.5|):")
conf = np.abs(probs - 0.5)
for lo, hi in [(0,.05),(.05,.1),(.1,.2),(.2,.5)]:
    m = (conf >= lo) & (conf < hi)
    if m.sum() < 20:
        continue
    print(f"  conf [{lo:.2f},{hi:.2f}): n={m.sum():>7,}  acc={((probs[m]>0.5)==labels[m]).mean()*100:.1f}%")
print("  ^ accuracy should RISE with confidence. If flat, confidence is decoration.")

results = dict(run_tag=RUN_TAG, use_mmt=USE_MMT_FEATURES, n_features=len(feature_cols),
               accuracy=float(acc), base_rate=float(base_rate), brier=brier, ece=float(ece))
print(f"\n>>> record this: {results}")

## 7. The Ablation

This is the whole point of the MMT subscription.

1. Set `USE_MMT_FEATURES = False` in section 0, **Restart Kernel**, Run All. Record the numbers.
2. Set `USE_MMT_FEATURES = True`, **Restart Kernel**, Run All. Record again.
3. Paste both into the cell below.

Read it honestly: a gain of a fraction of a percentage point on one split is noise.
What counts is a gain that shows up in **both** accuracy and calibration, and holds
in **both** H1 and H2.

In [ ]:
# Fill these in after running both configurations.
ohlcv_only = dict(accuracy=None, base_rate=None, brier=None, ece=None, n_features=None)
with_mmt   = dict(accuracy=None, base_rate=None, brier=None, ece=None, n_features=None)

if ohlcv_only["accuracy"] is not None and with_mmt["accuracy"] is not None:
    print(f"{'metric':<14}{'OHLCV only':>14}{'+ MMT flow':>14}{'delta':>12}")
    print("-" * 54)
    for k, better in [("accuracy", "higher"), ("brier", "lower"), ("ece", "lower")]:
        a, b = ohlcv_only[k], with_mmt[k]
        print(f"{k:<14}{a:>14.4f}{b:>14.4f}{b-a:>+12.4f}   ({better} is better)")
    print(f"{'n_features':<14}{ohlcv_only['n_features']:>14}{with_mmt['n_features']:>14}")
    print("\nVerdict: order flow earns its keep only if accuracy rises AND "
          "Brier/ECE fall, in both H1 and H2.")
else:
    print("Run both configurations first, then fill in the dicts above.")

## 8. Export Classifier

In [ ]:
os.makedirs("models", exist_ok=True)
import pickle
local_path = f"models/direction_classifier_{RUN_TAG}.pt"

torch.save({
    "state_dict": best_cls.state_dict(),
    "hparams": dict(best_cls.hparams),
    "feature_cols": feature_cols,
    "scaler_mean": scaler.mean_,
    "scaler_scale": scaler.scale_,          # MUST ship with the model
    "horizon": HORIZON,
    "encoder_length": MAX_ENCODER_LENGTH,
    "use_mmt_features": USE_MMT_FEATURES,
    "results": results,
    "data_start": data_start, "data_end": data_end,
    "trained_at": datetime.utcnow().isoformat(),
    "symbol": SYMBOL,
}, local_path)
print(f"Saved: {local_path}")
print("NOTE: inference must apply the SAME scaler (mean/scale saved above).")

UPLOAD_TO_GCS = False
if UPLOAD_TO_GCS:
    bucket = storage.Client(project=BQ_PROJECT).bucket(GCS_BUCKET)
    bucket.blob(f"direction_classifier_{RUN_TAG}.pt").upload_from_filename(local_path)
    print(f"Uploaded to gs://{GCS_BUCKET}/direction_classifier_{RUN_TAG}.pt")

## 9. (Optional) Legacy TFT Regression

V1's approach: predict the price path with quantile loss. Confirmed unprofitable,
kept only for reference / stop-loss placement via Q10-Q90. **Very slow on CPU**
(~500k rows). Set `RUN_TFT = True` in section 0 to enable.

In [ ]:
if not RUN_TFT:
    print("RUN_TFT = False — skipping the TFT regression. (Set it True in section 0 to run.)")
else:
    from pytorch_forecasting import TimeSeriesDataSet, GroupNormalizer, TemporalFusionTransformer
    from pytorch_forecasting.metrics import MultiHorizonMetric

    class DirectionAwareQuantileLoss(MultiHorizonMetric):
        def __init__(self, quantile_weight=0.7, direction_weight=0.3,
                     quantiles=[0.02, 0.1, 0.25, 0.5, 0.75, 0.9, 0.98], **kw):
            super().__init__(quantiles=quantiles, **kw)
            self.quantile_weight = quantile_weight
            self.direction_weight = direction_weight
            self.quantiles = quantiles

        def loss(self, y_pred, target):
            ls = []
            for i, q in enumerate(self.quantiles):
                e = target - y_pred[..., i]
                ls.append(torch.max((q - 1) * e, q * e).unsqueeze(-1))
            ql = torch.cat(ls, dim=-1).mean(dim=-1)
            pm = y_pred[..., len(self.quantiles) // 2]
            agree = torch.tanh((pm[:, 1:] - pm[:, :-1]) * 10) * torch.tanh((target[:, 1:] - target[:, :-1]) * 10)
            pen = torch.clamp(1.0 - agree, min=0.0) / 2.0
            pen = torch.cat([torch.zeros_like(pen[:, :1]), pen], dim=1)
            return self.quantile_weight * ql + self.direction_weight * pen

    df_tft = df_all.copy()
    known = [c for c in ["hour_sin","hour_cos","minute_sin","minute_cos","day_sin","day_cos"] if c in df_tft.columns]
    unknown = ["close"] + [c for c in feature_cols if c not in known]

    training = TimeSeriesDataSet(
        df_tft[df_tft.time_idx <= cutoff], time_idx="time_idx", target="close",
        group_ids=["group"], min_encoder_length=MAX_ENCODER_LENGTH // 2,
        max_encoder_length=MAX_ENCODER_LENGTH, max_prediction_length=MAX_PREDICTION_LENGTH,
        static_categoricals=["group"], time_varying_known_reals=known,
        time_varying_unknown_reals=unknown,
        target_normalizer=GroupNormalizer(groups=["group"], transformation="softplus"),
        add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True)
    validation = TimeSeriesDataSet.from_dataset(training, df_tft[df_tft.time_idx > cutoff], predict=False)

    tft = TemporalFusionTransformer.from_dataset(
        training, learning_rate=LEARNING_RATE, hidden_size=HIDDEN_SIZE,
        attention_head_size=ATTENTION_HEAD_SIZE, dropout=DROPOUT,
        hidden_continuous_size=HIDDEN_CONTINUOUS_SIZE, output_size=7,
        loss=DirectionAwareQuantileLoss(), reduce_on_plateau_patience=4)
    print(f"TFT parameters: {tft.size()/1e3:.1f}k")

    os.makedirs("checkpoints/training", exist_ok=True)
    tft_ckpt = ModelCheckpoint(dirpath="checkpoints/training", monitor="val_loss",
                               filename=f"tft-{RUN_TAG}" + "-{epoch:02d}-{val_loss:.4f}",
                               save_top_k=1, mode="min")
    trainer = pl.Trainer(max_epochs=MAX_EPOCHS,
        callbacks=[EarlyStopping(monitor="val_loss", patience=PATIENCE, mode="min"),
                   tft_ckpt, LearningRateMonitor(logging_interval="epoch")],
        accelerator=ACCELERATOR, gradient_clip_val=0.1)
    trainer.fit(tft,
                train_dataloaders=training.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=0),
                val_dataloaders=validation.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0))
    print(f"Best: {tft_ckpt.best_model_path}")